# Clip strategy sample test

Small RunPod test for comparing center-based 5-second clips and YOLO/ByteTrack-based 5-second clips before full training.

In [ ]:
from pathlib import Path
import csv
import json
import shutil
import subprocess
import sys
from collections import Counter, defaultdict
from IPython.display import Video, display

RUNPOD_ROOT = Path("/workspace/SKN27-FINAL-3Team")
PROJECT_ROOT = None
if RUNPOD_ROOT.exists() and (RUNPOD_ROOT / "requirements.txt").exists():
    PROJECT_ROOT = RUNPOD_ROOT
else:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "requirements.txt").exists() and (candidate / "ai").exists() and (candidate / "storage").exists():
            PROJECT_ROOT = candidate
            break
if PROJECT_ROOT is None:
    raise FileNotFoundError("project root not found")
MANIFEST_DIR = PROJECT_ROOT / "storage/vision/datasets/classification/manifests"
RAW_VIDEO_DIR = PROJECT_ROOT / "storage/vision/datasets/classification/raw_videos"
CLIP_TEST_DIR = PROJECT_ROOT / "storage/vision/datasets/classification/clip_strategy_test"
SAMPLE_MANIFEST = MANIFEST_DIR / "sample_700_coarse_manifest.csv"
FULL_DOWNLOAD_MANIFEST = MANIFEST_DIR / "train_700_download_manifest.csv"
TEST_DOWNLOAD_MANIFEST = MANIFEST_DIR / "sample_clip_strategy_download_manifest.csv"
CENTER_MANIFEST = MANIFEST_DIR / "sample_clip_strategy_center_5s_manifest.csv"
TRACK_MANIFEST = MANIFEST_DIR / "sample_clip_strategy_yolo_track_5s_manifest.csv"

SAMPLE_PER_LABEL = 1
SEED = 42
print("PROJECT_ROOT:", PROJECT_ROOT)


## Command helper

In [ ]:
def run_command(command, *, timeout=None):
    print(chr(10) + "$", " ".join(map(str, command)))
    completed = subprocess.run(list(map(str, command)), cwd=PROJECT_ROOT, text=True, capture_output=True, timeout=timeout)
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    completed.check_returncode()
    return completed


## Environment check

In [ ]:
run_command([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], timeout=3600)
usage = shutil.disk_usage(PROJECT_ROOT)
print("free_gb:", round(usage.free / 1024**3, 2))
run_command([sys.executable, "-c", "import cv2, torch; print('cv2 ok'); print('cuda_available:', torch.cuda.is_available()); print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)"])


## Build tiny test download manifest

Use existing downloaded videos if available. If not, download only SAMPLE_PER_LABEL videos per coarse label.

In [ ]:
def read_csv(path):
    with path.open("r", encoding="utf-8", newline="") as f:
        return list(csv.DictReader(f))


def write_csv(rows, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    fields = list(dict.fromkeys(key for row in rows for key in row.keys()))
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)


def ensure_sample_manifest():
    if SAMPLE_MANIFEST.exists():
        return

    classification_manifest = MANIFEST_DIR / "classification_manifest.csv"
    listing_path = PROJECT_ROOT / "storage/vision/manifests/drive_listing_aihub.json"

    if not classification_manifest.exists():
        if not listing_path.exists():
            raise FileNotFoundError(f"Drive listing not found: {listing_path}")
        run_command([
            sys.executable,
            "etl/vision/build_classification_manifest.py",
            "--listing", listing_path,
            "--output", classification_manifest,
        ], timeout=None)

    run_command([
        sys.executable,
        "etl/vision/sample_classification_dataset.py",
        "--input", classification_manifest,
        "--output", SAMPLE_MANIFEST,
        "--label-column", "coarse_label",
        "--per-label", "700",
        "--seed", str(SEED),
    ], timeout=None)


def select_per_label(rows, per_label):
    selected = []
    counts = defaultdict(int)
    for row in rows:
        label = row.get("coarse_label") or row.get("label") or "unknown"
        local_path = row.get("local_path") or row.get("file_path")
        if not local_path:
            continue
        path = Path(local_path)
        if not path.is_absolute():
            path = PROJECT_ROOT / path
        if not path.exists():
            continue
        if counts[label] >= per_label:
            continue
        copied = dict(row)
        copied["local_path"] = path.as_posix()
        copied["file_exists"] = "True"
        selected.append(copied)
        counts[label] += 1
    return selected


ensure_sample_manifest()

sample_rows = []
if FULL_DOWNLOAD_MANIFEST.exists():
    rows = read_csv(FULL_DOWNLOAD_MANIFEST)
    sample_rows = select_per_label(rows, SAMPLE_PER_LABEL)
    if sample_rows:
        write_csv(sample_rows, TEST_DOWNLOAD_MANIFEST)
    else:
        print("Full download manifest exists, but local videos are missing. Downloading sample videos again.")

if not sample_rows:
    run_command([
        sys.executable,
        "etl/vision/download_sampled_media.py",
        "--input", SAMPLE_MANIFEST,
        "--output", TEST_DOWNLOAD_MANIFEST,
        "--download-dir", RAW_VIDEO_DIR,
        "--label-column", "coarse_label",
        "--per-label", str(SAMPLE_PER_LABEL),
        "--split", "",
    ], timeout=None)
    sample_rows = read_csv(TEST_DOWNLOAD_MANIFEST)

print("test_download_manifest:", TEST_DOWNLOAD_MANIFEST)
print("rows:", len(sample_rows))
print("label_counts:", dict(Counter(row.get("coarse_label") for row in sample_rows)))
for row in sample_rows:
    print(row.get("coarse_label"), row.get("asset_id"), row.get("local_path"))


## Build center-based 5-second clips

In [ ]:
run_command([
    sys.executable,
    "etl/vision/build_training_clips.py",
    "--input", TEST_DOWNLOAD_MANIFEST,
    "--output", CENTER_MANIFEST,
    "--clip-dir", CLIP_TEST_DIR / "center",
    "--label-column", "coarse_label",
    "--clip-sec", "5",
    "--accident-source", "center",
    "--overwrite",
], timeout=None)

center_rows = read_csv(CENTER_MANIFEST)
print("center_rows:", len(center_rows))
print("status:", dict(Counter(row.get("clip_status") for row in center_rows)))


## Build YOLO/ByteTrack-based 5-second clips

This is intentionally small. If it is too slow, stop here and use center clips for the first full training run.

In [ ]:
run_command([
    sys.executable,
    "etl/vision/build_training_clips.py",
    "--input", TEST_DOWNLOAD_MANIFEST,
    "--output", TRACK_MANIFEST,
    "--clip-dir", CLIP_TEST_DIR / "yolo_track",
    "--label-column", "coarse_label",
    "--clip-sec", "5",
    "--accident-source", "yolo_track",
    "--model-name", "yolov8n.pt",
    "--overwrite",
], timeout=None)

track_rows = read_csv(TRACK_MANIFEST)
print("track_rows:", len(track_rows))
print("status:", dict(Counter(row.get("clip_status") for row in track_rows)))
print("basis:", dict(Counter(row.get("clip_basis") for row in track_rows)))


## Compare center vs YOLO/ByteTrack windows

In [ ]:
center_by_asset = {row.get("asset_id"): row for row in center_rows}
track_by_asset = {row.get("asset_id"): row for row in track_rows}

for asset_id, center in center_by_asset.items():
    track = track_by_asset.get(asset_id)
    if not track:
        continue
    print(chr(10) + "asset_id:", asset_id)
    print("label:", center.get("coarse_label"))
    print("center:", center.get("clip_start_sec"), "~", center.get("clip_end_sec"), "accident=", center.get("accident_candidate_sec"), "status=", center.get("clip_status"))
    print("track :", track.get("clip_start_sec"), "~", track.get("clip_end_sec"), "accident=", track.get("accident_candidate_sec"), "basis=", track.get("clip_basis"), "status=", track.get("clip_status"))


## Preview generated clips

In [ ]:
from IPython.display import Video, display
import imageio_ffmpeg
import subprocess
from pathlib import Path

ffmpeg_path = imageio_ffmpeg.get_ffmpeg_exe()


def to_h264_mp4(video_path):
    video_path = Path(video_path)
    h264_path = video_path.with_name(video_path.stem + "_h264.mp4")

    if h264_path.exists() and h264_path.stat().st_size > 0:
        return h264_path

    command = [
        ffmpeg_path,
        "-y",
        "-i", str(video_path),
        "-vcodec", "libx264",
        "-pix_fmt", "yuv420p",
        "-an",
        str(h264_path),
    ]
    result = subprocess.run(command, text=True, capture_output=True)
    print("h264_returncode:", result.returncode)
    if result.returncode != 0:
        print(result.stderr)
        return video_path
    return h264_path


for label, rows in [("center", center_rows), ("yolo_track", track_rows)]:
    print("\n##", label)
    for row in rows[:4]:
        clip_path = Path(row.get("local_path", ""))
        if not clip_path.is_absolute():
            clip_path = PROJECT_ROOT / clip_path
        print(row.get("coarse_label"), row.get("asset_id"), clip_path, clip_path.exists())
        if clip_path.exists():
            display_path = to_h264_mp4(clip_path)
            print("display_path:", display_path)
            display(Video(str(display_path), embed=True, width=480))


## Decision note

In [ ]:
print("If yolo_track windows look better and runtime is acceptable, use yolo_track for the next sample batch.")
print("If yolo_track is slow or unstable, use center clips for full VideoMAE training and keep yolo_track for validation/demo samples.")
